# 05 — Correção de amplitude no OD da estação EF01 (CETESB)

**Objetivo:** quebrar a régua do OD (sazonal-naive 0,1525 rolante / 0,1550 holdout) atacando o modo de falha documentado em `02b`/`03b`/`04b`: fase certa, **amplitude subestimada** quando a onda de julho cresce. Mesmo desenho do 00b–04b (janelas L=8640/H=288, split 70/15/15 + holdout de 10 dias, 10 origens diárias, segmento limpo 01/06 → 21/07).
**Braços:** (i) **saz_escalado** — sazonal-naive reescalado pela amplitude do contexto (analítico, sem treino); (ii) **lgbm_mult** — 288 LightGBM prevendo correção **multiplicativa** `Y/snaive` + features de amplitude + peso por recência; (iii) **dlw** — DLinear no resíduo com loss ponderada por recência + aumento de escala (jitter); (iv) **lstnet (02b)** recarregado (só inferência); (v) **ens** — NNLS dos cinco, pesos fitados só na val.
**Dados:** `dados/ef01-mogi-das-cruzes_oxigenio-dissolvido_2026-06-01_a_2026-08-31.csv` — ver `dados/README.md`.

In [ ]:
import json
import pickle
import random
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = Path.cwd() if (Path.cwd() / "dados").exists() else Path.cwd().parent
CSV = ROOT / "dados" / "ef01-mogi-das-cruzes_oxigenio-dissolvido_2026-06-01_a_2026-08-31.csv"
OUT = ROOT / "resultados" / "05-amplitude-od"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# --- protocolo travado (igual ao 00b) ---
L, H = 8640, 288
SEASON = 288
INTERP_LIMIT = 24
SEG_FIM = "2026-07-21 01:05"  # fim do segmento limpo (início do gap de 16,4 dias)
HOLDOUT_DIAS = 10
# --- 05: correção de amplitude ---
LN = 2016                  # contexto p/ dlw (últimos passos de X)
CTX = 2016                 # janela de amplitude (7 dias)
LGB_EST, LGB_LR, LGB_LEAVES = 150, 0.05, 31
LGB_STRIDE = 2
DL_EPOCHS, DL_PAT = 30, 5
TRAIN_STRIDE, ENS_STRIDE = 4, 4
W_ALPHA = 3.0              # peso de recência: w = 1 + ALPHA * posição normalizada (janelas tardias até 4x)
JIT_LO, JIT_HI = 0.8, 1.25  # jitter de escala no treino do dlw
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
print("ROOT:", ROOT, "| CSV existe:", CSV.exists(), "| torch:", torch.__version__)

## 1. Carga
Formato CETESB: `;`, decimal com vírgula, `windows-1252`, linha 1 = validação, linha 2 = cabeçalho.

In [ ]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
colvar = [c for c in df.columns if c != "Data hora"][0]
df = df.rename(columns={"Data hora": "ds", colvar: "y"}).sort_values("ds").reset_index(drop=True)
print(colvar, "|", df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()

## 2. EDA — perfil, o gap de 16 dias e ciclo diário

In [ ]:
isna = df["y"].isna().to_numpy()
bounds = np.where(np.diff(np.concatenate([[False], isna, [False]])))[0]
runs = sorted([(bounds[i], bounds[i+1]-1) for i in range(0, len(bounds), 2)],
              key=lambda r: r[1]-r[0], reverse=True)
print("top 5 gaps:")
for a, b in runs[:5]:
    print(f"  {df.ds[a]} → {df.ds[b]}  ({(b-a+1)*5/60:.1f} h)")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.4)
ax[0].axvspan(pd.Timestamp("2026-07-21 01:10"), pd.Timestamp("2026-08-06 11:30"),
              color="r", alpha=0.2, label="sensor morto (16,4 dias)")
ax[0].set_title("OD EF01 — série completa (faixa vermelha = gap, fora do experimento)")
ax[0].set_ylabel("OD (mg/L)")
ax[0].legend(fontsize=8)
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição do OD")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("OD por hora do dia (ciclo diário?)")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva:", OUT / "figs" / "01-eda.png")

## 3. Limpeza + recorte do segmento limpo
Grade de 5 min, interpolação máx. 2 h e **corte em 21/07 01:05** (antes do gap). Tudo a jusante usa só o segmento 01/06 → 21/07.

In [ ]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_full = df.set_index("ds")["y"].reindex(idx)
s = s_full.loc[:SEG_FIM].interpolate(method="time", limit=INTERP_LIMIT)
print(f"segmento: {s.index.min()} → {s.index.max()} ({len(s)} slots = {len(s)*5/60/24:.1f} dias)")
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")

amostra = slice("2026-06-08", "2026-06-15")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_full[amostra].index, s_full[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 08–15/06")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")

## 4. Estacionariedade (ADF) e decomposição STL
Idêntico ao 00b (últimos 4032 pontos do treino, período 288).

In [ ]:
n_total = len(s)
n_train = int(n_total * 0.70)
train = s.iloc[:n_train].dropna()
stat, pval, *_ = adfuller(train.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(train.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")

## 5. Janelamento + holdout puro
Amostras `(L=8640 → H=288)` por janela deslizante, só janelas 100% observadas. Pré-holdout: split 70/15/15 **sem shuffle**. Holdout: últimos 10 dias + 10 origens diárias. **Idêntico ao 00b**.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
n = len(X)
ZONE = s.index.max() - pd.Timedelta(days=HOLDOUT_DIAS)
is_hold = ends >= (ZONE + pd.Timedelta(minutes=5 * (H - 1)))
ho = np.where(is_hold)[0]
pre = np.where(~is_hold)[0]
i1, i2 = int(len(pre) * 0.70), int(len(pre) * 0.85)
tr, va, te = pre[:i1], pre[i1:i2], pre[i2:]
splits = {"train": tr, "val": va, "test": te, "holdout": ho}
for k, idx in splits.items():
    print(f"{k}: {len(idx)} janelas | alvos {ends[idx[0]].date()} → {ends[idx[-1]].date()}")
print(f"janelas descartadas (com NaN): {len(s) - L - H + 1 - n}")
print(f"zona holdout (alvos): {ZONE.date()} → {s.index.max().date()}")
daily_ends = [ZONE + pd.Timedelta(minutes=5 * (H - 1 + H * k)) for k in range(HOLDOUT_DIAS)]
daily_idx = np.array([int(np.where(ends == d)[0][0]) for d in daily_ends])
print("dias previstos:", [str(ends[i].date()) for i in daily_idx])
TR_END = ends[tr[-1]]

## 6. Baselines baratos + sazonal reescalado (analítico)
Persistência, sazonal-naive (lag 288) e média móvel 288 — vetorizados, mesmos do 00b — mais o **saz_escalado**: o template sazonal recentrado na média do contexto e com desvios multiplicados por `std(ctx)/std(ref)` (clip [0,5, 2,0]). Só usa passado: `ctx = X[:, -2016:]`, `ref = X[:, L-288-2016:L-288]`.

In [ ]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

def saz_escalado(X_):
    S = np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1)
    Ctx = X_[:, -CTX:]
    Ref = X_[:, L - SEASON - CTX:L - SEASON]
    mu_c = Ctx.mean(axis=1, keepdims=True)
    mu_r = Ref.mean(axis=1, keepdims=True)
    sc = Ctx.std(axis=1, keepdims=True) / np.maximum(Ref.std(axis=1, keepdims=True), 1e-6)
    sc = np.clip(sc, 0.5, 2.0)
    return (mu_c + (S - mu_r) * sc).astype(np.float64)

Xte, Yte = X[te], Y[te]
Xho, Yho = X[ho], Y[ho]
pred_te = cheap_preds(Xte)
pred_te["saz_escalado"] = saz_escalado(Xte)
print("teste rolante (baratos):")
print(pd.DataFrame({m: metricas(Yte, p) for m, p in pred_te.items()}).T.round(4).to_string())

## 7. LightGBM multiplicativo (um modelo por passo do horizonte)
Alvo = razão `q = Y / snaive(X)` (~1,0; OD sempre > 0) em vez do resíduo aditivo: quando a amplitude cresce, a correção escala junto. Features = as 25 do 04b + 4 de amplitude do contexto (`ctx_std`, `ctx_ptp`, `seas_ptp7`, `amp_ratio`, tudo causal) + hora-do-dia do passo-alvo. Treino com peso de recência `w = 1 + 3·posição` (janelas tardias = amplitude maior, até 4x). 288 `LGBMRegressor` (150 árvores). Modelos salvos em `modelos/lgbm_mult_steps.pkl` (gitignore, >100 MB).

In [ ]:
import lightgbm as lgb

def base_feats(Xb, E):
    cols = [Xb[:, -k] for k in [1, 2, 3, 6, 12, 24, 36, 72, 144, 287, 288, 289, 576, 2016]]
    phase = np.stack([Xb[:, L - 288*k] for k in range(1, 8)], axis=1)
    cols += [phase.mean(1), phase.std(1)]
    for w in [12, 36, 144, 288]:
        cols += [Xb[:, -w:].mean(1), Xb[:, -w:].std(1)]
    cols += [Xb[:, -2016:].mean(1)]
    F = np.stack(cols, axis=1)
    em = (E.hour.to_numpy()*60 + E.minute.to_numpy()).astype(np.float32)
    return F.astype(np.float32), em, phase

def amp_feats(Xb, phase):
    Ctx = Xb[:, -CTX:]
    Ref = Xb[:, L - SEASON - CTX:L - SEASON]
    ctx_std = Ctx.std(axis=1)
    ctx_ptp = Ctx.max(axis=1) - Ctx.min(axis=1)
    seas_ptp = phase.max(axis=1) - phase.min(axis=1)
    ratio = ctx_std / np.maximum(Ref.std(axis=1), 1e-6)
    return np.stack([ctx_std, ctx_ptp, seas_ptp, ratio], axis=1).astype(np.float32)

def hour_sincos(em, j):
    hh = ((em - (H - 1 - j)*5) % 1440 // 60).astype(np.float32)
    return np.sin(2*np.pi*hh/24).astype(np.float32), np.cos(2*np.pi*hh/24).astype(np.float32)

tr2 = tr[::LGB_STRIDE]
Xb_tr = X[tr2]  # hoist: avaliado 1x (sem isso a compreensão abaixo retinha 288 cópias, OOM)
Ftr, emtr, Phtr = base_feats(Xb_tr, ends[tr2])
Atr = amp_feats(Xb_tr, Phtr)
Ftr = np.column_stack([Ftr, Atr])
Str = np.stack([Xb_tr[:, L - SEASON + h] for h in range(H)], axis=1)  # piso sazonal
Qtr = np.clip((Y[tr2] / np.maximum(Str, 1e-6)).astype(np.float32), 0.5, 1.5)  # razão: o que as árvores aprendem
w_tr = (1.0 + W_ALPHA * np.linspace(0, 1, len(tr2))).astype(np.float32)  # recência (tardias até 4x)
print(f"features: {Ftr.shape} + hora do passo | razão média: {Qtr.mean():.4f} std: {Qtr.std():.4f}")

models = []
t0 = time.time()
for j in range(H):
    sh, ch = hour_sincos(emtr, j)
    m = lgb.LGBMRegressor(n_estimators=LGB_EST, learning_rate=LGB_LR, num_leaves=LGB_LEAVES,
                          verbosity=-1, force_col_wise=True)
    m.fit(np.column_stack([Ftr, sh, ch]), Qtr[:, j], sample_weight=w_tr)
    models.append(m)
    if (j + 1) % 72 == 0:
        print(f"  lgbm_mult {j+1}/{H} ...", flush=True)
print(f"lgbm_mult: {len(models)} modelos em {time.time()-t0:.0f}s")
with open(OUT / "modelos" / "lgbm_mult_steps.pkl", "wb") as f:
    pickle.dump(models, f)
print("modelos salvos: modelos/lgbm_mult_steps.pkl")

def prevê_lgbm_mult(idxs):
    ii = np.asarray(idxs)
    Xb = X[ii]  # hoist: avaliado 1x (OOM se dentro da compreensão)
    F, em, Ph = base_feats(Xb, ends[ii])
    F = np.column_stack([F, amp_feats(Xb, Ph)])
    S = np.stack([Xb[:, L - SEASON + h] for h in range(H)], axis=1)
    P = np.empty((len(ii), H), dtype=np.float32)
    for j, m in enumerate(models):
        sh, ch = hour_sincos(em, j)
        P[:, j] = S[:, j] * np.clip(m.predict(np.column_stack([F, sh, ch])), 0.5, 1.5)
    return P

imp = np.mean([m.booster_.feature_importance(importance_type="gain") for m in models], axis=0)
nomes = ["lag1", "lag2", "lag3", "lag6", "lag12", "lag24", "lag36", "lag72", "lag144",
         "lag287", "lag288", "lag289", "lag576", "lag2016", "seasmean7", "seasstd7",
         "rm12", "rs12", "rm36", "rs36", "rm144", "rs144", "rm288", "rs288", "rm2016",
         "ctx_std", "ctx_ptp", "seas_ptp7", "amp_ratio",
         "hora_sin", "hora_cos"]
ordem = np.argsort(imp)[::-1]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh([nomes[k] for k in ordem], imp[ordem])
ax.set_title("LightGBM-mult — importância média das features (gain, 288 modelos)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "07-importancia-lgbm.png")
print("top features:", [(nomes[k], round(float(imp[k]), 1)) for k in ordem[:5]])
print("fig salva: 07-importancia-lgbm.png")

## 8. DLinear ponderado + régua LSTNet + ensemble NNLS
**dlw** — mesmo DLinear-5min do 04b no resíduo `r = Y − snaive(X)`, com duas mudanças: (i) loss MSE ponderada por recência (mesmo `w` do §7); (ii) aumento de escala autoconsistente no treino — `X' = mu + f·(X−mu)`, `R' = f·R` com `f ~ U(0,8, 1,25)` (deriva de `snaive(X') = mu + f·(snaive(X)−mu)`, logo o resíduo escala por `f`). Régua do 02b por checkpoint. Pesos do ensemble via NNLS nas janelas da val sobre 5 colunas (sazonal, saz_escalado, lstnet, lgbm_mult, dlw).

In [ ]:
def snaive(X_):
    return np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1)

class DLinearLite(nn.Module):
    def __init__(self, k=25):
        super().__init__()
        self.pool = nn.AvgPool1d(k, stride=1, padding=k // 2)
        self.lin_t = nn.Linear(LN, H)
        self.lin_s = nn.Linear(LN, H)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        t = self.pool(xn.unsqueeze(1)).squeeze(1)
        y = self.lin_t(t) + self.lin_s(xn - t)
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg  # sem +mu: alvo é resíduo de média ~0

def monta_res(idxs):
    ii = np.asarray(idxs)
    return X[ii][:, -LN:].astype(np.float32), (Y[ii] - snaive(X[ii])).astype(np.float32)

Xr_tr, Rr_tr = monta_res(tr[::TRAIN_STRIDE])
Xr_va, Rr_va = monta_res(va[::TRAIN_STRIDE])
w_tr_dl = (1.0 + W_ALPHA * np.linspace(0, 1, len(Xr_tr))).astype(np.float32)
print(f"residual: treino {Xr_tr.shape} val {Xr_va.shape}")
dlw = DLinearLite().to(DEVICE)
opt = torch.optim.Adam(dlw.parameters(), lr=1e-3)
tr_loader = DataLoader(TensorDataset(torch.from_numpy(Xr_tr), torch.from_numpy(Rr_tr),
                                     torch.from_numpy(w_tr_dl)), batch_size=512, shuffle=True)
va_loader = DataLoader(TensorDataset(torch.from_numpy(Xr_va), torch.from_numpy(Rr_va)), batch_size=512)
best, patience = float("inf"), 0
t0 = time.time()
rng = np.random.default_rng(SEED)
for ep in range(1, DL_EPOCHS + 1):
    dlw.train()
    for xb, yb, wb in tr_loader:
        mu = xb.mean(dim=1, keepdim=True)
        f = torch.from_numpy(rng.uniform(JIT_LO, JIT_HI, size=(len(xb), 1)).astype(np.float32))
        xb_a = mu + f * (xb - mu)  # jitter de escala autoconsistente
        yb_a = f * yb
        opt.zero_grad()
        se = ((dlw(xb_a) - yb_a) ** 2).mean(dim=1)
        loss = (se * wb) / wb.mean()  # ponderada por recência
        loss.mean().backward(); opt.step()
    dlw.eval(); vl = 0.0
    with torch.no_grad():
        for xb, yb in va_loader:
            vl += float((((dlw(xb) - yb) ** 2).mean()).item()) * len(xb)
    vl /= len(va_loader.dataset)
    tag = ""
    if vl < best:
        best, patience = vl, 0
        torch.save({"state": dlw.state_dict()}, OUT / "modelos" / "dlinear_w_od.pt")
        tag = " *"
    else:
        patience += 1
    print(f"dlw ep {ep:02d} val={vl:.5f}{tag}", flush=True)
    if patience >= DL_PAT:
        break
print(f"dlw em {time.time()-t0:.0f}s | melhor val={best:.5f}")
dlw.load_state_dict(torch.load(OUT / "modelos" / "dlinear_w_od.pt", map_location="cpu", weights_only=False)["state"])
dlw.eval()

# régua 02b (LSTNet, só inferência)
class LSTNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(3, 32, kernel_size=12, stride=6)
        self.gru = nn.GRU(32, 64, batch_first=True)
        self.skipcell = nn.GRUCell(32, 32)
        self.head = nn.Linear(96, 288)
        self.ar = nn.Linear(288, 288)
        self.drop = nn.Dropout(0.1)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, xv, tod):
        mu = xv.mean(dim=1, keepdim=True); sg = xv.std(dim=1, keepdim=True).clamp_min(1e-3)
        vn = self.gamma * (xv - mu) / sg + self.beta
        f = self.drop(torch.relu(self.conv(torch.cat([vn.unsqueeze(1), tod.transpose(1, 2)], dim=1))))
        f = f.transpose(1, 2)
        _, h = self.gru(f)
        B, T, _ = f.shape
        hs = torch.zeros(B, 32, device=f.device)
        states = [hs]
        for t in range(T):
            prev = states[t - 48] if t - 48 >= 0 else states[0]
            hs = self.skipcell(f[:, t, :], prev)
            states.append(hs)
        g = self.gamma.clamp_min(1e-3)
        yn = self.head(self.drop(torch.cat([h.squeeze(0), hs], dim=1)))
        ya = self.ar(vn[:, -288:])
        return (yn + ya - self.beta) / g * sg + mu

ckpt02 = torch.load(ROOT / "resultados" / "02b-lstnet-od" / "modelos" / "lstnet_od.pt",
                    map_location="cpu", weights_only=False)
ruler = LSTNet1D().to(DEVICE)
ruler.load_state_dict(ckpt02["state"])
ruler.eval()
SIN5 = np.sin(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
COS5 = np.cos(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
Tln = sliding_window_view(np.stack([SIN5, COS5], axis=1), 2016, axis=0).transpose(0, 2, 1).astype(np.float32)
pos_end = s.index.get_indexer(ends - pd.Timedelta(minutes=5*H))
rowln = pos_end - 2016 + 1

@torch.no_grad()
def prevê_tudo(idxs, batch=256):
    ii = np.asarray(idxs)
    Ps = snaive(X[ii])
    Se = saz_escalado(X[ii])
    Gm = prevê_lgbm_mult(ii)
    outs = []
    Xt = torch.from_numpy(X[ii][:, -2016:].astype(np.float32))
    for b in range(0, len(Xt), batch):
        outs.append(dlw(Xt[b:b+batch]).numpy())
    Dw = Ps + np.concatenate(outs)
    return Ps, Se, Gm, Dw

# inferência da régua (precisa do contexto 2016 + ToD)
Wln2 = sliding_window_view(s.to_numpy().astype(np.float32), 2016)
@torch.no_grad()
def prevê_ruler(idxs, batch=256):
    ii = np.asarray(idxs)
    outs = []
    for b in range(0, len(ii), batch):
        xb = torch.from_numpy(Wln2[rowln[ii[b:b+batch]]])
        tb = torch.from_numpy(Tln[rowln[ii[b:b+batch]]])
        outs.append(ruler(xb, tb).numpy())
    return np.concatenate(outs)

# pesos NNLS na val
from scipy.optimize import nnls
va2 = va[::ENS_STRIDE]
Ps_v, Se_v, Gm_v, Dw_v = prevê_tudo(va2)
Pn_v = prevê_ruler(va2)
A = np.column_stack([Ps_v.ravel(), Se_v.ravel(), Pn_v.ravel(), Gm_v.ravel(), Dw_v.ravel()])
w, _ = nnls(A, Y[va2].ravel())
pesos = {k: round(float(v), 4) for k, v in zip(["sazonal", "saz_esc", "lstnet", "lgbm_mult", "dlw"], w)}
json.dump({"pesos": pesos, "mode": "nnls-ensemble sobre sazonal+saz_esc+lstnet+lgbm_mult+dlw"},
          open(OUT / "modelos" / "ensemble.json", "w"))
json.dump({"mode": "saz-escalado + lgbm-mult-288 + dlinear-ponderado + nnls-ensemble", "LN": LN, "CTX": CTX},
          open(OUT / "modelos" / "normalizacao.json", "w"))
print("pesos ensemble (nnls na val):", pesos)

def ensemble(Ps, Se, Pn, Gm, Dw):
    return w[0]*Ps + w[1]*Se + w[2]*Pn + w[3]*Gm + w[4]*Dw

t0 = time.time()
Ps_te, Se_te, Gm_te, Dw_te = prevê_tudo(te); Pn_te = prevê_ruler(te)
En_te = ensemble(Ps_te, Se_te, Pn_te, Gm_te, Dw_te)
Ps_ho, Se_ho, Gm_ho, Dw_ho = prevê_tudo(ho); Pn_ho = prevê_ruler(ho)
Ps_d, Se_d, Gm_d, Dw_d = prevê_tudo(daily_idx); Pn_d = prevê_ruler(daily_idx)
En_d = ensemble(Ps_d, Se_d, Pn_d, Gm_d, Dw_d)
print(f"inferência em {time.time()-t0:.0f}s")
print("SAZ_ESC teste:", {k: round(v, 4) for k, v in metricas(Yte, Se_te).items()})
print("LGBM_MULT teste:", {k: round(v, 4) for k, v in metricas(Yte, Gm_te).items()})
print("DLW teste:", {k: round(v, 4) for k, v in metricas(Yte, Dw_te).items()})
print("ENS teste:", {k: round(v, 4) for k, v in metricas(Yte, En_te).items()})
print("ENS holdout:", {k: round(v, 4) for k, v in metricas(Y[daily_idx], En_d).items()})

## 9. Comparação final + holdout dia a dia
Tabela do teste rolante (todas as origens), tabela do holdout diário (10 dias) e MAE por dia. Réguas do 00b impressas para referência (sazonal-naive 0,1525 / 0,1550).

In [ ]:
linhas = {m: metricas(Yte, p) for m, p in pred_te.items()}
linhas["lstnet"] = metricas(Yte, Pn_te)
linhas["lgbm_mult"] = metricas(Yte, Gm_te)
linhas["dlw"] = metricas(Yte, Dw_te)
linhas["ens"] = metricas(Yte, En_te)
tab = pd.DataFrame(linhas).T.round(4)
tab.to_csv(OUT / "metricas_baseline.csv")
print("=== teste rolante ===")
print(tab.to_string())

Yd = Y[daily_idx]
ch_d = cheap_preds(X[daily_idx])
diario = {m: metricas(Yd, ch_d[m]) for m in ["persistencia", "sazonal_naive_288", "media_movel_288"]}
diario["saz_escalado"] = metricas(Yd, Se_d)
diario["lstnet"] = metricas(Yd, Pn_d)
diario["lgbm_mult"] = metricas(Yd, Gm_d)
diario["dlw"] = metricas(Yd, Dw_d)
diario["ens"] = metricas(Yd, En_d)
tab_d = pd.DataFrame(diario).T.round(4)
tab_d.to_csv(OUT / "metricas_holdout.csv")
print("=== holdout diário (10 dias) ===")
print(tab_d.to_string())

por_dia = pd.DataFrame(
    {"persistencia": [mae(Yd[k:k+1], ch_d["persistencia"][k:k+1]) for k in range(len(Yd))],
     "sazonal_naive_288": [mae(Yd[k:k+1], ch_d["sazonal_naive_288"][k:k+1]) for k in range(len(Yd))],
     "saz_escalado": [mae(Yd[k:k+1], Se_d[k:k+1]) for k in range(len(Yd))],
     "lstnet": [mae(Yd[k:k+1], Pn_d[k:k+1]) for k in range(len(Yd))],
     "lgbm_mult": [mae(Yd[k:k+1], Gm_d[k:k+1]) for k in range(len(Yd))],
     "dlw": [mae(Yd[k:k+1], Dw_d[k:k+1]) for k in range(len(Yd))],
     "ens": [mae(Yd[k:k+1], En_d[k:k+1]) for k in range(len(Yd))],
     "amp_dia": [(float(Yd[k].max() - Yd[k].min())) for k in range(len(Yd))]}, 
    index=[str(ends[i].date()) for i in daily_idx])
print(por_dia.round(4).to_string())
print(f"\nRégua 00b (teste rolante): sazonal-naive = 0.1525 | este exp: {tab['MAE'].idxmin()} = {tab['MAE'].min():.4f}")
print(f"Régua 00b (holdout diário): sazonal-naive = 0.1550 | este exp: {tab_d['MAE'].idxmin()} = {tab_d['MAE'].min():.4f}")
print(f"pesos ensemble: {pesos}")
por_dia.to_csv(OUT / "metricas_por_dia.csv")

In [ ]:
E = ends[te]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
for ax, k in zip(axes, [0, len(Xte)//2, -1]):
    tc = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H+2015)), E[k] - pd.Timedelta(minutes=5*H), freq="5min")
    ax.plot(tc, Xte[k][-2016:], lw=0.8, label="contexto (cauda 7d)")
    tf = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H-1)), E[k], freq="5min")
    ax.plot(tf, Yte[k], "k-", lw=1.5, label="real")
    ax.plot(tf, pred_te["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, Se_te[k], lw=1, alpha=0.9, label="saz_escalado")
    ax.plot(tf, Pn_te[k], lw=1, alpha=0.6, label="lstnet(02b)")
    ax.plot(tf, Gm_te[k], lw=1, alpha=0.9, label="lgbm_mult")
    ax.plot(tf, En_te[k], lw=1.2, alpha=0.9, label="ens")
    ax.set_title(f"origem {E[k]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

fig, ax = plt.subplots(figsize=(8, 4))
tab["MAE"].sort_values().plot.barh(ax=ax)
ax.set_title("MAE no teste rolante (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

fig, axes = plt.subplots(5, 2, figsize=(14, 12), sharey=False)
for ax, k in zip(axes.ravel(), range(len(Yd))):
    tf = pd.date_range(ends[daily_idx[k]] - pd.Timedelta(minutes=5*(H-1)), ends[daily_idx[k]], freq="5min")
    ax.plot(tf, Yd[k], "k-", lw=1.2, label="real")
    ax.plot(tf, ch_d["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, Se_d[k], lw=1, alpha=0.9, label="saz_escalado")
    ax.plot(tf, Pn_d[k], lw=1, alpha=0.6, label="lstnet(02b)")
    ax.plot(tf, Gm_d[k], lw=1, alpha=0.9, label="lgbm_mult")
    ax.plot(tf, En_d[k], lw=1.2, alpha=0.9, label="ens")
    ax.set_title(f"dia previsto {ends[daily_idx[k]].date()} (MAE ens={por_dia['ens'].iloc[k]:.3f} vs saz={por_dia['sazonal_naive_288'].iloc[k]:.3f})")
    ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-holdout-dias.png")

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].bar(range(len(Yd)), por_dia["amp_dia"].values)
axes[0].set_title("Amplitude do dia real (max-min, mg/L)")
axes[0].set_xticks(range(len(Yd)), [str(ends[i].date()) for i in daily_idx], rotation=30, fontsize=8)
for m in ["sazonal_naive_288", "saz_escalado", "lstnet", "lgbm_mult", "dlw", "ens"]:
    axes[1].plot(range(len(Yd)), por_dia[m].values, marker="o", ms=3, label=m)
axes[1].set_title("MAE por dia previsto (o erro cresce com a amplitude?)")
axes[1].set_xticks(range(len(Yd)), [str(ends[i].date()) for i in daily_idx], rotation=30, fontsize=8)
axes[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "08-erro-amplitude.png")
print("figs salvas")

## 10. Conclusões e próximos passos

- A régua do 00b (sazonal-naive 0,1525 / 0,1550) está impressa na §9 e o LSTNet do 02b vem recarregado por checkpoint na mesma tabela.
- Os três braços testam a mesma hipótese por vias diferentes: o erro do OD escala com a amplitude — `saz_escalado` (analítico), `lgbm_mult` (razão + features de amplitude + recência) e `dlw` (loss ponderada + jitter de escala).
- Pesos NNLS fitados só na val; se o `ens` (ou qualquer braço) vencer no holdout, vira a nova régua do OD.
- Se nada vencer: o próximo passo é saída probabilística (quantis) e/ou teste de transferência no pós-gap 06→31/08.
- Artefatos em `resultados/05-amplitude-od/`: `metricas_baseline.csv`, `metricas_holdout.csv`, `metricas_por_dia.csv`, `modelos/dlinear_w_od.pt`, `modelos/ensemble.json`, `modelos/normalizacao.json` e `figs/` (`lgbm_mult_steps.pkl` fora do git, regenerável).